# Bölüm 14 · Kovaryans Analizi (ANCOVA)

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b14/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b14/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b14/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b14/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b14/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b14-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b14-kovaryans-analizi'
modules=['uygulamalar/b14-kovaryans-analizi/hedef-dagilim']
expected_main=json.loads((root/"bolumler/b14/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 14 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b14/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 14 — görevler

Güncel Revizyon 17 görevleri; önce kendiniz çözün.

## D1–D3: Modelden Önce

1.  Bu veride bağımlı değişken, grup ve kovaryat hangileridir? Doza neden ön-test denemez?

2.  Her iki grupta ortak doz desteği var mı? 3 mg/günde karşılaştırma aynı güvenle yapılabilir mi?

3.  Kovaryatı yalnız p değerine göre seçmek neden uygun değildir?

# Rehberli ve Bağımsız Etkinlikler

## G1–G4: Rehberli Çözüm

1.  Ortak eğim modeliyle 1 mg/gündeki iki düzeltilmiş ortalamayı bulun. Ham fark neden değişmedi?

2.  Ortak ve ayrı eğim modellerinin F testini SSE ve hata sd üzerinden yeniden hesaplayın.

3.  $L=[0,1,0,x]$ kullanarak 1 mg/gündeki farkı, SH’yi ve Bonferroni aralığını hesaplayın.

4.  Ortak modelde grup için kısmi eta kareyi bulun; hangi varyans oranını temsil etmediğini açıklayın.

## B1–B3: Bağımsız Çözüm

1.  Merkez 2 mg/güne taşınırsa grup katsayısını bulun. Tahminler ve etkileşim testi değişir mi?

2.  Ayrı eğim modelinin hücre modeline göre doğrusal biçim testini hesaplayın. İki serbestlik derecesi nereden gelir?

3.  Hücre modelindeki 2 mg/gün farkını ve Bonferroni aralığını yorumlayın. HC3 ve tek-kayıt duyarlılığı aynı şey midir?

## H1: Hatalı Raporu Onarma

Altı iddiayı gerçek hesap, hedef karşılaştırma ve kanıt sınırı bakımından düzeltin.

## T1–T3: Hedef Dağılım ve Ön-Test Denetimi

1.  Üç hücre farkından eşit, düşük doz ve yüksek doz hedeflerinin ortalama farklarını hesaplayın. Bunlar neden aynı hedef için üç sağlamlık koşusu değildir? Ağırlıkları iki gruba farklı uygulamak hangi soruyu değiştirir?

2.  $L_w$ ile farkın SH’sini ve üç hedeflik aileyi açıklayın. Ortak eğim modelinde neden yalnız bir kontrast vardır? Hücre modelinin ortalama dozda tanımlanmaması standartlaştırılmış ortalamayı engeller mi?

3.  Yeni bir ön-test/son-test çalışmasında $Y\sim G+X$ ile $(Y-X)\sim G+X$ neden aynı grup katsayısını verir? Ön-testi ikinci modelden çıkarmak hangi kısıtı getirir? Bu örnekte dose neden X yerine kullanılamaz?

## P1: Ön-Test Kontrollü Yeni Bir Çalışma

Eğitim bağlamında gerçek bir ön-test kovaryatıyla yeni çalışma planlayın: hedef grup karşılaştırması, kovaryatın işlem öncesi ölçümü, güvenirlik, örtüşme, eğim/şekil kontrolü, referans değer, çoklu aile, eksikler ve raporlama kararlarını belirtin. ToothGrowth hayvan verisini öğrenci ön-test verisi diye yeniden etiketlemeyin. Yapılmamış rastgeleleştirme, veri toplama veya uzman görüşmesi “planlandı” olarak yazılmalıdır. Rubrik: kaynak/tasarım 20, model/hesap 30, kontrast/belirsizlik 25, yorum/sınırlılıklar 15, yeniden üretim 10 puan.

Yanıtlar Ek F, Bölüm 14’tedir (s. kitaptaki ilgili sayfa).

# Alıştırmalar

## Kolay Düzey

1.  ANCOVA ile ANOVA arasındaki temel fark nedir?

2.  Düzeltilmiş ortalama neyi ifade eder?

3.  Regresyon eğimlerinin homojenliği ne demektir?

4.  Kovaryat seçerken hangi koşul önemlidir?

5.  Kovaryat anlamlı, grup etkisi anlamsızsa ne düşünülür?

## Orta Düzey

1.  Grup $\times$ kovaryat etkileşimi anlamlıdır. Bu durumu açıklayınız.

2.  Ham ortalamalarda A grubu, düzeltilmiş ortalamalarda B grubu öne çıkmaktadır. Bu neyi gösterir?

3.  Ön-test puanı kovaryat seçilmiştir. Bu seçimin gücünü açıklayınız.

4.  İşlem sonrası bir değişken kovaryat seçilirse neden sorun olabilir?

5.  ANCOVA’da neden yalnız ham ortalamalar yerine düzeltilmiş ortalamalar raporlanmalıdır?

## İleri Düzey

1.  Kovaryat bağımlı değişkenle zayıf ilişkiliyse ANCOVA’nın yararı açısından ne anlama gelir?

2.  Eğim homojenliği bozulmuşsa hangi alternatifler veya uyarılar düşünülmelidir?

3.  Kovaryatta ciddi ölçüm hatası varsa yorum nasıl etkilenir?

4.  “ANCOVA tüm başlangıç farklarını tamamen yok etti” ifadesi neden fazla güçlüdür?

5.  Güçlü bir ANCOVA raporunda bulunması gereken en az beş unsuru yazınız.

Yöntem belgelerine erişim: 6 Eylül 2026.


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b14/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b14/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

In [ ]:
manifest={"chapter":14,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b14-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b14-kovaryans-analizi',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)